# P300 BCI Speller — Google Colab Step-by-Step Guide

> **What you will learn:** This notebook walks you through the complete P300 Speller
> pipeline, from generating synthetic EEG to decoding intended characters, all
> running in-browser with the built-in Simulator — no hardware required.

---

## What is the P300 ERP?

The **P300** is an event-related potential (ERP) — a positive voltage peak in the EEG
that appears ~300 ms after a rare or task-relevant stimulus. In a **P300 BCI speller**
(Farwell & Donchin, 1988):

1. A 6 × 6 letter matrix is shown on a screen.
2. Rows and columns flash in a random order.
3. The user **silently counts** flashes of their target letter.
4. Every time the target's row **or** column flashes, the brain produces a P300.
5. The system identifies which row and which column produced a P300 — their intersection
   is the intended letter.

```
┌─────────────────────────────────────────┐
│  A   B  [C]  D   E   F   ← target col  │
│  G   H   I   J   K   L                 │
│  M   N   O   P   Q   R                 │
│  S   T   U   V   W   X                 │
│  Y   Z   1   2   3   4                 │
│  5   6   7   8   9   _                 │
│ ──────── target row ─────────          │
└─────────────────────────────────────────┘
  Row 0 flashes → P300!
  Col 2 flashes → P300!
  Intersection = 'C' ✓
```

---

## Notebook Sections

| Section | Topic |
|---------|-------|
| 1 | Installation & Setup |
| 2 | System Architecture |
| 3 | Synthetic EEG Generation |
| 4 | Signal Processing (Filtering + Epoching) |
| 5 | ERP Visualization |
| 6 | Feature Extraction |
| 7 | LDA Classifier Training |
| 8 | Full Pipeline Demo (Calibrate + Spell) |
| 9 | Accuracy vs N_Sequences Analysis |
| 10 | Key Configuration Parameters |
| 11 | Real Hardware Setup |


---
## Section 1 — Installation & Setup

Run this cell once at the start of each Colab session.

In [ ]:
# ── 1a. Install Python dependencies ──────────────────────────────────────────
!pip install -q numpy scipy scikit-learn joblib pyyaml pygame pyserial \
               matplotlib seaborn

print("✓ Packages installed")

In [ ]:
# ── 1b. Clone the repository ──────────────────────────────────────────────────
import os

REPO_ROOT = '/content/P300-Analyzer-V1'

if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/waleed178-del/p300-analyzer-v1.git {REPO_ROOT}
else:
    print("Repo already cloned — pulling latest changes")
    !git -C {REPO_ROOT} pull --ff-only

print(f"\nRepo root: {REPO_ROOT}")
!ls {REPO_ROOT}

In [ ]:
# ── 1c. Configure Python path and headless display ───────────────────────────
import sys
import os

# Add the src/ directory so modules can be imported by bare name.
SRC_DIR = os.path.join(REPO_ROOT, 'p300_speller', 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# Tell pygame to use a dummy display driver — no physical screen needed.
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')
os.environ.setdefault('SDL_AUDIODRIVER', 'dummy')

# Work from the repo root so relative config paths resolve correctly.
os.chdir(REPO_ROOT)

print("Python path:", sys.path[0])
print("Working directory:", os.getcwd())
print("SDL_VIDEODRIVER:", os.environ.get('SDL_VIDEODRIVER'))

---
## Section 2 — System Architecture

```
┌──────────────────────────────────────────────────────────────────────────┐
│                         P300 Speller Pipeline                            │
│                                                                          │
│  ┌────────────┐    ┌─────────────┐    ┌──────────┐    ┌─────────────┐  │
│  │ Acquisition│───▶│  Processing  │───▶│ Features │───▶│ Classifier  │  │
│  │(Serial/Sim)│    │(BP+Notch+   │    │(CAR +    │    │(LDA/SVM +   │  │
│  │ 250 Hz EEG │    │ Epoch+Base) │    │ Decimate+│    │ argmax row/ │  │
│  └────────────┘    └─────────────┘    │ Flatten) │    │ col scores) │  │
│         ▲                             └──────────┘    └─────────────┘  │
│         │ markers                                                        │
│  ┌──────┴──────┐                                                         │
│  │   Stimulus  │  6×6 grid, random row/col flash order                  │
│  │  (pygame)   │  perf_counter timestamps on every flash                │
│  └─────────────┘                                                         │
│         ▲                                                                │
│  ┌──────┴──────────────────────────────┐                                 │
│  │           SessionManager            │  operator prompts, logging      │
│  │  calibrate()  ──▶  free_spell()     │                                 │
│  └─────────────────────────────────────┘                                 │
└──────────────────────────────────────────────────────────────────────────┘
```

**File map:**
```
p300_speller/
├── configs/config.yaml        ← every tunable parameter lives here
├── src/
│   ├── acquisition.py         ← SerialAcquisition  +  Simulator
│   ├── processing.py          ← bandpass, notch, epoching, baseline
│   ├── features.py            ← CAR, decimate, flatten → feature vector
│   ├── classifier.py          ← StandardScaler + LDA/SVM, save/load
│   ├── stimulus.py            ← pygame 6×6 grid, row/col flashing
│   ├── session.py             ← SessionManager (prompts + logging)
│   └── main_pipeline.py       ← P300Pipeline: train + spell
├── arduino/
│   └── p300_eeg_acquisition.ino  ← ADS1115 firmware
├── tests/                     ← pytest unit tests
└── run.py                     ← CLI entry point
```

---
## Section 3 — Synthetic EEG Generation

The `Simulator` class produces realistic multi-channel EEG: broadband noise + a
10 Hz alpha rhythm. When the *attended* row or column flashes, it schedules a
Gaussian P300 bump at 300 ms post-stimulus.

We'll visualise the raw synthetic signal to build intuition before any processing.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import time

from acquisition import Simulator, StimulusMarker

FS       = 250.0          # Hz
CHANNELS = ['Fz', 'Cz', 'Pz']

# ── Start the simulator and let the buffer prime ──────────────────────────
sim = Simulator(
    sampling_rate_hz   = FS,
    channel_names      = CHANNELS,
    ring_buffer_s      = 10.0,
    noise_uv           = 12.0,
    p300_amplitude_uv  = 6.0,
    p300_latency_ms    = 300.0,
    p300_width_ms      = 120.0,
    alpha_amplitude_uv = 8.0,
    seed               = 42,
)

# Target: 'A' is at row 0, col 0 in the default matrix.
sim.set_target(row=0, col=0)
sim.start()
time.sleep(0.5)   # let the buffer fill

# ── Push two target flashes and some non-target flashes ──────────────────
flash_times = []
flash_kinds = []

def push_flash(kind, index, is_target=False):
    t = time.perf_counter()
    marker = StimulusMarker(code=(index + 1) if kind == 'row' else (6 + index + 1),
                             kind=kind, index=index, perf_time=t)
    sim.push_marker(marker)
    flash_times.append(t)
    flash_kinds.append(('target' if is_target else 'non-target', kind, index))
    time.sleep(0.175)   # 100ms flash + 75ms ISI

# Simulate one sequence: row 0 is target row, col 0 is target col.
push_flash('row', 1)           # non-target row
push_flash('col', 3)           # non-target col
push_flash('row', 0, True)     # TARGET row → P300 in ~300ms
push_flash('col', 2)           # non-target col
push_flash('col', 0, True)     # TARGET col → P300 in ~300ms
push_flash('row', 3)           # non-target row

time.sleep(1.2)   # wait for the last P300 to fully develop

ts, eeg = sim.get_buffer()
sim.stop()

print(f"Buffer: {len(ts)} samples  ({len(ts)/FS:.1f} s)  ·  {eeg.shape[1]} channels")

In [ ]:
# ── Plot the raw EEG with flash markers ───────────────────────────────────
t_rel = ts - ts[0]   # seconds since start of buffer

fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
colors = {'#2196F3': 0, '#9C27B0': 1, '#F44336': 2}
ch_colors = ['#2196F3', '#9C27B0', '#F44336']

for i, (ax, ch, col) in enumerate(zip(axes, CHANNELS, ch_colors)):
    ax.plot(t_rel, eeg[:, i], color=col, linewidth=0.7, alpha=0.85)
    ax.set_ylabel(f'{ch} (μV)', fontsize=9)
    ax.grid(True, alpha=0.25)

# Overlay flash markers on all subplots
for ft, (label, kind, idx) in zip(flash_times, flash_kinds):
    ft_rel = ft - ts[0]
    is_tgt = label == 'target'
    lc  = '#FF5722' if is_tgt else '#607D8B'
    lw  = 2.0 if is_tgt else 0.8
    ls  = '-'  if is_tgt else '--'
    for ax in axes:
        ax.axvline(ft_rel, color=lc, linewidth=lw, linestyle=ls, alpha=0.9)
        if is_tgt:
            ax.axvline(ft_rel + 0.3, color='#4CAF50', linewidth=1.0,
                       linestyle=':', alpha=0.8)   # expected P300 peak

# Legend on the top subplot
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], color='#FF5722', lw=2, label='Target flash'),
    Line2D([0], [0], color='#607D8B', lw=0.8, ls='--', label='Non-target flash'),
    Line2D([0], [0], color='#4CAF50', lw=1,  ls=':',  label='Expected P300 (+300 ms)'),
]
axes[0].legend(handles=legend_handles, loc='upper right', fontsize=8)
axes[-1].set_xlabel('Time (s)', fontsize=10)
fig.suptitle('Synthetic EEG — Raw Trace with P300 Markers\n'
             '(3-channel, 250 Hz, α-rhythm + broadband noise)', fontsize=12)
plt.tight_layout()
plt.show()

print("NOTE: P300 peaks (green dotted) appear ~300 ms after each target flash.")
print("They are buried in noise in single trials — averaging many epochs reveals them.")

---
## Section 4 — Signal Processing Pipeline

Three steps clean the raw EEG before epoching:

| Step | Purpose | Parameters |
|------|---------|------------|
| **Bandpass** | Remove DC drift and muscle artefacts above 30 Hz | 0.5–30 Hz, Butterworth order 4 |
| **Notch** | Suppress mains-hum at 50 Hz (60 Hz in NA) | Q = 30 |
| **Epoching** | Slice continuous signal into flash-aligned windows | −100 to +800 ms |
| **Baseline** | Remove pre-stimulus mean so epochs share a common zero | −100 to 0 ms |

All filtering is **zero-phase** (`filtfilt` / `sosfiltfilt`) so the P300 latency
is not distorted.

In [ ]:
from processing import apply_bandpass, apply_notch, epoch_data, precondition

# ── Step 1 & 2: Band-pass + notch ────────────────────────────────────────
eeg_bp     = apply_bandpass(eeg, low_hz=0.5, high_hz=30.0, fs=FS, order=4)
eeg_clean  = apply_notch(eeg_bp, freq_hz=50.0, fs=FS, quality_factor=30.0)

# ── Visualise one channel before vs after ────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 7), sharex=True)
titles = ['Raw EEG (Pz)', 'After Bandpass 0.5–30 Hz (Pz)', 'After Notch 50 Hz (Pz)']
signals = [eeg[:, 2], eeg_bp[:, 2], eeg_clean[:, 2]]

for ax, title, sig in zip(axes, titles, signals):
    ax.plot(t_rel, sig, linewidth=0.7, color='#F44336')
    ax.set_ylabel('μV', fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.grid(True, alpha=0.25)
    for ft, (label, *_) in zip(flash_times, flash_kinds):
        ft_rel = ft - ts[0]
        is_tgt = label == 'target'
        ax.axvline(ft_rel, color='#FF5722' if is_tgt else '#607D8B',
                   lw=1.5 if is_tgt else 0.6,
                   ls='-'  if is_tgt else '--', alpha=0.7)

axes[-1].set_xlabel('Time (s)', fontsize=10)
fig.suptitle('Spectral Conditioning — Pz Channel', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 3 & 4: Epoch + baseline ─────────────────────────────────────────
# Re-collect a full dataset with many more epochs for a clean ERP.
# We run the simulator for 10 sequences (~20 s) to gather enough trials.

print("Collecting 10 sequences of synthetic EEG (~20 s)...")

N_SEQUENCES  = 10
N_ROWS       = 6
N_COLS       = 6
FLASH_MS     = 100
ISI_MS       = 75
TARGET_ROW   = 0      # 'A' is row 0, col 0
TARGET_COL   = 0

sim2 = Simulator(
    sampling_rate_hz=FS, channel_names=CHANNELS, ring_buffer_s=30.0,
    noise_uv=12.0, p300_amplitude_uv=6.0,
    p300_latency_ms=300.0, p300_width_ms=120.0,
    alpha_amplitude_uv=8.0, seed=42)
sim2.set_target(row=TARGET_ROW, col=TARGET_COL)
sim2.start()
time.sleep(0.5)

all_markers = []
rng = np.random.default_rng(7)
stimuli = [('row', r) for r in range(N_ROWS)] + [('col', c) for c in range(N_COLS)]

for seq in range(N_SEQUENCES):
    order = rng.permutation(len(stimuli)).tolist()
    for idx in order:
        kind, index = stimuli[idx]
        code = (index + 1) if kind == 'row' else (N_ROWS + index + 1)
        perf_t = time.perf_counter()
        m = StimulusMarker(code=code, kind=kind, index=index, perf_time=perf_t)
        sim2.push_marker(m)
        all_markers.append(m)
        time.sleep((FLASH_MS + ISI_MS) / 1000.0)

time.sleep(1.2)   # post-roll: last epoch must fully develop
ts2, eeg2 = sim2.get_buffer()
sim2.stop()

# Condition
eeg2_clean = apply_bandpass(eeg2, 0.5, 30.0, FS, 4)
eeg2_clean = apply_notch(eeg2_clean, 50.0, FS, 30.0)

# Epoch
epochs = epoch_data(ts2, eeg2_clean, all_markers, FS,
                    tmin_s=-0.1, tmax_s=0.8, baseline=(-0.1, 0.0))

target_epochs     = [e for e in epochs
                     if (e.marker.kind == 'row' and e.marker.index == TARGET_ROW)
                     or (e.marker.kind == 'col' and e.marker.index == TARGET_COL)]
nontarget_epochs  = [e for e in epochs
                     if not ((e.marker.kind == 'row' and e.marker.index == TARGET_ROW)
                             or (e.marker.kind == 'col' and e.marker.index == TARGET_COL))]

print(f"Total epochs  : {len(epochs)}")
print(f"Target epochs : {len(target_epochs)}")
print(f"Non-target    : {len(nontarget_epochs)}")
print(f"Epoch shape   : {epochs[0].data.shape}  (n_times × n_channels)")

---
## Section 5 — ERP Visualization

A single-trial P300 is buried in noise — only **averaging** across many trials
reveals the characteristic positive deflection. The plot below shows the
**grand average** across all target and non-target epochs.

In [ ]:
# ── Grand averages ────────────────────────────────────────────────────────
target_avg    = np.mean([e.data for e in target_epochs],    axis=0)   # (n_times, 3)
nontarget_avg = np.mean([e.data for e in nontarget_epochs], axis=0)
target_sem    = np.std([e.data for e in target_epochs],    axis=0) / np.sqrt(len(target_epochs))
nontarget_sem = np.std([e.data for e in nontarget_epochs], axis=0) / np.sqrt(len(nontarget_epochs))
times = epochs[0].times

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for i, (ax, ch) in enumerate(zip(axes, CHANNELS)):
    # Non-target
    ax.plot(times, nontarget_avg[:, i], '#E53935', linewidth=1.8, label=f'Non-target (n={len(nontarget_epochs)})')
    ax.fill_between(times,
                    nontarget_avg[:, i] - nontarget_sem[:, i],
                    nontarget_avg[:, i] + nontarget_sem[:, i],
                    color='#E53935', alpha=0.15)
    # Target
    ax.plot(times, target_avg[:, i], '#1E88E5', linewidth=2.2, label=f'Target (n={len(target_epochs)})')
    ax.fill_between(times,
                    target_avg[:, i] - target_sem[:, i],
                    target_avg[:, i] + target_sem[:, i],
                    color='#1E88E5', alpha=0.18)
    # Annotations
    ax.axvline(0.0, color='k', linewidth=0.8, linestyle='-',  label='Flash onset')
    ax.axvline(0.3, color='#4CAF50', linewidth=1.5, linestyle=':', label='P300 peak (300 ms)')
    ax.axhline(0.0, color='k', linewidth=0.4, linestyle=':')
    ax.set_title(f'Channel {ch}', fontsize=12)
    ax.set_xlabel('Time relative to flash (s)', fontsize=9)
    ax.set_xlim(-0.1, 0.8)
    ax.grid(True, alpha=0.25)
    if i == 0:
        ax.set_ylabel('Amplitude (μV)', fontsize=10)
        ax.legend(fontsize=8)

fig.suptitle('Event-Related Potential (ERP)\n'
             'Grand Average — Target vs Non-target Flashes', fontsize=13)
plt.tight_layout()
plt.show()

# ── Single-trial overlay on Pz to show noise ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, epoch_list, label, color in zip(
        axes,
        [target_epochs, nontarget_epochs],
        ['Target epochs (Pz)', 'Non-target epochs (Pz)'],
        ['#1E88E5', '#E53935']):
    for e in epoch_list[:15]:    # show first 15 trials
        ax.plot(times, e.data[:, 2], color=color, alpha=0.25, linewidth=0.6)
    avg = np.mean([e.data[:, 2] for e in epoch_list], axis=0)
    ax.plot(times, avg, color=color, linewidth=2.5, label='Average')
    ax.axvline(0, color='k', linewidth=0.8)
    ax.axvline(0.3, color='#4CAF50', linewidth=1.5, ls=':', label='300 ms')
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Time (s)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25)
axes[0].set_ylabel('μV')
fig.suptitle('Single-trial Overlay — P300 is hidden in noise per trial,\n'
             'but emerges in the average', fontsize=11)
plt.tight_layout()
plt.show()

---
## Section 6 — Feature Extraction

The classifier needs a fixed-length numeric vector. Three deterministic steps
convert each epoch:

1. **Spatial filter** (`none` or `car`) — common-average reference for dense
   montages; set to `none` here because 3-channel CAR would destroy signal.
2. **Temporal decimation** — block-average from 250 Hz to 20 Hz (factor 12).
   The P300 is slow (~0.3 s) and doesn't need high temporal resolution.
3. **Flatten** — ravel the `(n_times_ds, n_channels)` array to a 1-D vector
   `[t0_ch0, t0_ch1, t0_ch2, t1_ch0, ...]`.

In [ ]:
from features import epochs_to_matrix, epoch_to_features

feat_cfg = {'downsample_hz': 20, 'spatial_filter': 'none'}

# ── Build design matrix ───────────────────────────────────────────────────
X_target    = epochs_to_matrix(target_epochs,    FS, feat_cfg)
X_nontarget = epochs_to_matrix(nontarget_epochs, FS, feat_cfg)

print(f"Target feature matrix    : {X_target.shape}")
print(f"Non-target feature matrix: {X_nontarget.shape}")

# Feature vector breakdown
n_times_ds = X_target.shape[1] // len(CHANNELS)
print(f"\nEach feature vector = {n_times_ds} time points × {len(CHANNELS)} channels"
      f" = {X_target.shape[1]} features")
print(f"(250 Hz × 0.9 s = 225 samples, decimated by 12 → {225 // 12} points per channel)")

In [ ]:
# ── Visualise feature vectors ─────────────────────────────────────────────
# Approximate time axis for decimated epochs (start of each 12-sample block)
factor = int(round(FS / 20.0))   # 12
n_ds = 225 // factor              # 18 time points per channel
times_ds = times[::factor][:n_ds]

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# Top: average feature vectors reshaped back to (time, channels)
avg_target_feat    = X_target.mean(axis=0).reshape(n_ds, len(CHANNELS))
avg_nontarget_feat = X_nontarget.mean(axis=0).reshape(n_ds, len(CHANNELS))

for i, (ch, col) in enumerate(zip(CHANNELS, ['#2196F3', '#9C27B0', '#F44336'])):
    axes[0].plot(times_ds, avg_target_feat[:, i],    color=col, ls='-',  lw=2,   label=f'{ch} (target)')
    axes[0].plot(times_ds, avg_nontarget_feat[:, i], color=col, ls='--', lw=1.2, label=f'{ch} (non-target)')

axes[0].axvline(0.3, color='#4CAF50', ls=':', lw=1.5, label='300 ms')
axes[0].set_title('Average Feature Vector (after 20 Hz decimation)', fontsize=11)
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('μV')
axes[0].legend(fontsize=7, ncol=4)
axes[0].grid(True, alpha=0.25)

# Bottom: heatmap of feature matrices
n_show = min(30, len(X_target), len(X_nontarget))
heat = np.vstack([X_target[:n_show], X_nontarget[:n_show]])
vmax = np.percentile(np.abs(heat), 95)
im = axes[1].imshow(heat, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].axhline(n_show - 0.5, color='k', lw=2)
axes[1].set_yticks([n_show // 2, n_show + n_show // 2])
axes[1].set_yticklabels(['Target', 'Non-target'], fontsize=9)
axes[1].set_xlabel('Feature index  (time × channel, flattened)', fontsize=9)
axes[1].set_title(f'Feature Matrix Heatmap — first {n_show} epochs each class', fontsize=11)
plt.colorbar(im, ax=axes[1], label='μV')

plt.tight_layout()
plt.show()

---
## Section 7 — LDA Classifier Training

**Linear Discriminant Analysis (LDA)** with Ledoit-Wolf shrinkage is the
workhorse of P300 BCIs. It is:
- **Closed-form** (no gradient descent, no learning rate).
- **Regularised** (shrinkage handles the high-dimensional, few-sample regime).
- **Interpretable** — the weight vector is directly the spatial-temporal filter
  that maximises the signal-to-noise ratio.

The standard LDA decision function produces a real-valued *score* (signed distance
to the boundary). Scores are **averaged** across the repeated flashes of each row
and each column, and the `argmax` of row-mean and column-mean scores gives the
predicted letter.

In [ ]:
from classifier import P300Classifier, NON_TARGET, TARGET, decode_character_scores
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# ── Build labelled design matrix ──────────────────────────────────────────
X_all = epochs_to_matrix(epochs, FS, feat_cfg)
y_all = np.array(
    [TARGET if ((e.marker.kind == 'row'  and e.marker.index == TARGET_ROW) or
                (e.marker.kind == 'col'  and e.marker.index == TARGET_COL))
     else NON_TARGET
     for e in epochs],
    dtype=int
)

print(f"Design matrix : {X_all.shape}")
print(f"Class balance : {np.bincount(y_all)}  (non-target / target)")

# ── Train ─────────────────────────────────────────────────────────────────
clf = P300Classifier(model_type='lda', lda_shrinkage='auto', class_weight='balanced')
report = clf.fit(X_all, y_all, cv_folds=5)

print(f"\n{'─'*55}")
print(f"Training epochs : {report.n_samples}  (targets: {report.n_targets})")
print(f"Resubstitution accuracy : {report.train_accuracy:.3f}")
print(f"Cross-validated AUC     : {report.cv_auc_mean:.3f} ± {report.cv_auc_std:.3f}")
print(f"{'─'*55}")
print("CV AUC ≈ 1.0 on synthetic data — real EEG typically gives 0.6–0.85.")

In [ ]:
# ── Confusion matrix + score histogram ───────────────────────────────────
scores = clf.decision_scores(X_all)
preds  = clf.predict(X_all)
cm     = confusion_matrix(y_all, preds)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Non-target', 'Target'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (resubstitution)', fontsize=11)

# Score distribution
bins = np.linspace(scores.min(), scores.max(), 50)
axes[1].hist(scores[y_all == NON_TARGET], bins=bins, alpha=0.7,
             color='#E53935', label=f'Non-target (n={(y_all==NON_TARGET).sum()})')
axes[1].hist(scores[y_all == TARGET],     bins=bins, alpha=0.7,
             color='#1E88E5', label=f'Target (n={(y_all==TARGET).sum()})')
axes[1].axvline(0, color='k', lw=1.5, ls='--', label='Decision boundary')
axes[1].set_xlabel('LDA Decision Score', fontsize=10)
axes[1].set_ylabel('Count', fontsize=10)
axes[1].set_title('LDA Score Distributions', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

---
## Section 8 — Full Pipeline Demo (Calibrate + Spell)

This section runs the complete `selftest` command:
1. **Calibrate** — the system trains on the word `CAT` (3 characters, 8 sequences each).
2. **Spell** — the trained model tries to decode `CAT` from fresh flashes.

> ⏱ **Expected runtime: ~2 minutes** (real-time stimulus presentation at 250 Hz).
> You can reduce it by lowering `flash_duration_ms` and `inter_stimulus_interval_ms`
> in the config, but faster flashes give less reliable P300 in real hardware.

The selftest forces `use_simulator: true` and `headless: true`, so no EEG
hardware or physical display is required.

In [ ]:
import subprocess

env = {**os.environ, 'SDL_VIDEODRIVER': 'dummy', 'SDL_AUDIODRIVER': 'dummy'}

print("Running: python p300_speller/run.py selftest")
print("(This runs in real-time — expect ~2 minutes)\n")

result = subprocess.run(
    ['python', 'p300_speller/run.py', 'selftest'],
    capture_output=True, text=True, env=env, cwd=REPO_ROOT
)

# Print output live
for line in result.stdout.splitlines():
    print(line)

if result.returncode == 0:
    print("\n✓ Selftest PASSED")
else:
    print("\n✗ Selftest FAILED (return code", result.returncode, ")")
    print(result.stderr)

In [ ]:
# ── You can also drive the pipeline via the Python API directly ───────────
import yaml
from main_pipeline import P300Pipeline, load_config
from session import SessionManager

config = load_config('p300_speller/configs/config.yaml')

# Force simulator + headless
config['acquisition']['use_simulator'] = True
config['speller']['headless']          = True
config['speller']['inter_character_pause_s'] = 0.1
# Absolute model path so we don't depend on CWD
config['classifier']['model_path'] = os.path.join(
    REPO_ROOT, 'p300_speller', 'models', 'p300_model_demo.joblib')
config['session']['output_text_path'] = os.path.join(
    REPO_ROOT, 'p300_speller', 'output', 'demo_spelled.txt')

session = SessionManager(config)

print("Calibrating on 'HI' (2 characters, 8 sequences each)...")
report = session.calibrate(words=['HI'], n_sequences=8)
print("Training complete:", report)

print("\nSpelling 'HI'...")
decoded = session.free_spell(n_characters=2, n_sequences=8, simulated_intent='HI')
print(f"\nDecoded text: {decoded!r}  {'✓' if decoded == 'HI' else '✗'}")

---
## Section 9 — Accuracy vs. N_Sequences

Single-trial P300 detection is noisy (SNR ~0.5 for typical amplitudes). By
**averaging** multiple flash repetitions per character the noise averages out
while the signal grows, giving accuracy that improves with `n_sequences`.

This section simulates the accuracy curve by collecting many epochs once,
then evaluating the decoder at different levels of averaging — much faster than
re-running the pipeline for each `n_sequences` value.

In [ ]:
# ── Collect 20 sequences of data for target = 'A' (row 0, col 0) ─────────
MAX_SEQ = 20

print(f"Collecting {MAX_SEQ} sequences (~{MAX_SEQ * 12 * 0.175:.0f} s)...")

sim3 = Simulator(
    sampling_rate_hz=FS, channel_names=CHANNELS, ring_buffer_s=60.0,
    noise_uv=12.0, p300_amplitude_uv=6.0,
    p300_latency_ms=300.0, p300_width_ms=120.0,
    alpha_amplitude_uv=8.0, seed=99)
sim3.set_target(row=0, col=0)
sim3.start()
time.sleep(0.5)

markers_acc = []
rng3 = np.random.default_rng(13)
for seq in range(MAX_SEQ):
    # Use index-based permutation so mixed-type tuples are not coerced by numpy.
    idx_order = rng3.permutation(len(stimuli)).tolist()
    for i in idx_order:
        kind, index = stimuli[i]
        code = (index + 1) if kind == 'row' else (N_ROWS + index + 1)
        perf_t = time.perf_counter()
        m = StimulusMarker(code=code, kind=kind, index=index, perf_time=perf_t)
        sim3.push_marker(m)
        markers_acc.append(m)
        time.sleep(0.175)

time.sleep(1.2)
ts3, eeg3 = sim3.get_buffer()
sim3.stop()

eeg3_clean = apply_bandpass(eeg3, 0.5, 30.0, FS, 4)
eeg3_clean = apply_notch(eeg3_clean, 50.0, FS, 30.0)
epochs3 = epoch_data(ts3, eeg3_clean, markers_acc, FS,
                     tmin_s=-0.1, tmax_s=0.8, baseline=(-0.1, 0.0))

print(f"Collected {len(epochs3)} epochs for {MAX_SEQ} sequences")

In [ ]:
# ── Simulate decoding at different n_sequences ────────────────────────────
# Group epochs by (kind, index) so we can select n per row/col easily.
from collections import defaultdict

ep_by_stimulus = defaultdict(list)
for ep in epochs3:
    ep_by_stimulus[(ep.marker.kind, ep.marker.index)].append(ep)

# Re-use the same trained classifier (fitted on the earlier 10-sequence data).
# In a real sweep you'd re-train for each n_sequences, but the clf is fixed here
# so we measure the *decoding* accuracy as a function of averaging depth.

N_TRIALS = 20    # repeat the decoding trial with different random subsets
n_seq_values = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20]
rng_s = np.random.default_rng(0)

accuracy_means = []
accuracy_stds  = []

for n_seq in n_seq_values:
    correct_counts = []
    for _ in range(N_TRIALS):
        row_sum = np.zeros(N_ROWS)
        row_cnt = np.zeros(N_ROWS)
        col_sum = np.zeros(N_COLS)
        col_cnt = np.zeros(N_COLS)

        for kind in ('row', 'col'):
            n_items = N_ROWS if kind == 'row' else N_COLS
            for idx in range(n_items):
                pool = ep_by_stimulus[(kind, idx)]
                chosen = rng_s.choice(len(pool), size=min(n_seq, len(pool)), replace=False)
                subset = [pool[j] for j in chosen]
                if not subset:
                    continue
                X_sub = epochs_to_matrix(subset, FS, feat_cfg)
                sc = clf.decision_scores(X_sub).mean()
                if kind == 'row':
                    row_sum[idx] += sc;  row_cnt[idx] += 1
                else:
                    col_sum[idx] += sc;  col_cnt[idx] += 1

        row_scores = row_sum / np.maximum(row_cnt, 1)
        col_scores = col_sum / np.maximum(col_cnt, 1)
        pred_row, pred_col = int(np.argmax(row_scores)), int(np.argmax(col_scores))
        correct_counts.append(int(pred_row == 0 and pred_col == 0))

    accuracy_means.append(np.mean(correct_counts))
    accuracy_stds.append(np.std(correct_counts) / np.sqrt(N_TRIALS))
    print(f"  n_sequences={n_seq:2d}  →  accuracy={accuracy_means[-1]:.0%}")

In [ ]:
# ── Plot accuracy vs n_sequences ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(n_seq_values, accuracy_means, 'o-', color='#1E88E5', linewidth=2.2,
        markersize=7, markerfacecolor='white', markeredgewidth=2, label='Decoding accuracy')
ax.fill_between(
    n_seq_values,
    [m - 2*s for m, s in zip(accuracy_means, accuracy_stds)],
    [m + 2*s for m, s in zip(accuracy_means, accuracy_stds)],
    color='#1E88E5', alpha=0.15, label='±2 SE'
)

# Chance level: 1/(6×6) = 1/36
ax.axhline(1/36, color='#E53935', ls=':', lw=1.5, label='Chance (1/36 ≈ 2.8%)')
ax.axhline(1.0,  color='#4CAF50', ls=':', lw=1.5, label='Perfect accuracy')

# Timing annotation (default config)
char_time = [n * 12 * 0.175 for n in n_seq_values]
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
ax2.set_xticks(n_seq_values[::2])
ax2.set_xticklabels([f'{t:.1f} s' for t in char_time[::2]], fontsize=8)
ax2.set_xlabel('Time per character (default timing: 100 ms flash + 75 ms ISI)', fontsize=9)

ax.set_xlabel('n_sequences (repetitions per character)', fontsize=11)
ax.set_ylabel('Decoding accuracy (simulator)', fontsize=11)
ax.set_ylim(-0.05, 1.15)
ax.set_xticks(n_seq_values)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
fig.suptitle('Accuracy vs. Number of Flash Repetitions\n'
             '(more sequences → better SNR via averaging, but slower spelling)', fontsize=12)
plt.tight_layout()
plt.show()

print("\nPractical recommendation:")
print("  ≥ 8 sequences for reliable spelling with a healthy, focused user.")
print("  ≥ 15 sequences for users with fatigue or reduced P300 amplitude.")

---
## Section 10 — Key Configuration Parameters

Everything lives in `p300_speller/configs/config.yaml`. Edit there — no Python
code changes required.

### Acquisition

| Parameter | Default | Effect |
|-----------|---------|--------|
| `sampling_rate_hz` | 250 | ADC rate; must match Arduino firmware |
| `channels` | `[Fz, Cz, Pz]` | EEG channel labels (order must match hardware) |
| `use_simulator` | `true` | `false` → connect real Arduino over Serial |
| `serial.port` | `/dev/ttyACM0` | Windows: `COM3`, macOS: `/dev/cu.usbmodemXXXX` |
| `simulator.p300_amplitude_uv` | 6.0 | Synthetic P300 peak amplitude |
| `simulator.noise_uv` | 12.0 | Background noise std-dev |

### Signal Processing

| Parameter | Default | Effect |
|-----------|---------|--------|
| `bandpass.low_hz` | 0.5 | High-pass cutoff — removes slow drift |
| `bandpass.high_hz` | 30.0 | Low-pass cutoff — removes muscle noise |
| `notch.freq_hz` | 50.0 | Mains hum (EU); 60.0 for North America |
| `epoch.tmin_s` | −0.1 | Pre-stimulus baseline window start |
| `epoch.tmax_s` | 0.8 | End of epoch; must cover P300 latency |

### Features & Classifier

| Parameter | Default | Effect |
|-----------|---------|--------|
| `features.downsample_hz` | 20 | Epoch decimation rate |
| `features.spatial_filter` | `none` | `car` for dense montages (>8 ch) |
| `classifier.model_type` | `lda` | `svm` for an alternative discriminator |
| `classifier.lda_shrinkage` | `auto` | Ledoit-Wolf regularisation |

### Speller & Session

| Parameter | Default | Effect |
|-----------|---------|--------|
| `speller.n_sequences` | 10 | Flash repetitions per character — trade speed vs accuracy |
| `speller.flash_duration_ms` | 100 | Duration of each row/column highlight |
| `speller.inter_stimulus_interval_ms` | 75 | Dark gap between flashes |
| `speller.headless` | `false` | `true` → dummy display (CI / servers) |
| `session.calibration_words` | `[THE, QUICK, BROWN, FOX]` | Words used during `train` phase |

---

### Switching from Simulator to Real Hardware

```yaml
# In configs/config.yaml:
acquisition:
  use_simulator: false          # ← switch to hardware
  serial:
    port: "/dev/ttyACM0"        # adjust to your port
    baud_rate: 115200
  channels: ["Pz"]             # single-channel build
```

Then run:
```bash
python p300_speller/run.py train
python p300_speller/run.py spell
```

---
## Section 11 — Real Hardware Setup

### Components

| Component | Purpose | Approx cost |
|-----------|---------|-------------|
| Arduino Uno | Host controller + Serial bridge | $10–20 |
| Adafruit ADS1115 (16-bit ADC, I²C) | EEG digitiser | $15–20 |
| Ag/AgCl cup electrodes | EEG recording sites (Pz, Cz, Fz) | $20–40 |
| Conductive electrode gel | Good skin contact | $10 |
| Ground + reference clip | Ear/mastoid reference | $5–10 |

### Wiring Diagram

```
Arduino Uno         ADS1115
─────────────       ──────────────
5V      ──────────▶ VDD
GND     ──────────▶ GND
A4 (SDA)──────────▶ SDA
A5 (SCL)──────────▶ SCL
                    A0 ──▶ Pz electrode (through 1 kΩ)
                    A1 ──▶ Cz electrode (optional)
                    A2 ──▶ Fz electrode (optional)
GND     ──────────▶ Reference electrode (ear/mastoid)
GND     ──────────▶ Ground electrode (forehead)

ADS1115 ADDR pin → GND  (I²C address 0x48)
```

### Firmware Flash

1. Open `p300_speller/arduino/p300_eeg_acquisition.ino` in the Arduino IDE.
2. Set `NUM_CHANNELS` to `1` (Pz only) or `3` (Fz, Cz, Pz).
3. Set `SAMPLE_RATE_HZ` to match `config.yaml` (default `250`).
4. Upload to the Uno (Board: *Arduino Uno*, Port: `COM3` / `/dev/ttyACM0`).

### Serial Protocol (reference)

The firmware streams ASCII lines at 115200 baud:

```
D,<seq>,<micros>,<marker>,<ch0>[,<ch1>,<ch2>]
```

The host sends `M<code>\n` sync commands to align neural data to flash onset.

### Safety Note

> ⚠️ This is **research / assistive** software. Clinical use requires:
> - Medical-grade, galvanically isolated EEG hardware.
> - Formal validation with the individual user.
> - An error-correction / undo protocol (a `_` backspace symbol is included).
> - Compliance with relevant regulations (FDA 510(k), CE Mark, etc.).

---

## Troubleshooting

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| `pygame.error: No available video device` | SDL driver not set | Add `os.environ['SDL_VIDEODRIVER'] = 'dummy'` before any import |
| `RuntimeError: Calibration produced no epochs` | Buffer too short / bad timing | Increase `ring_buffer_s`; check `sampling_rate_hz` matches hardware |
| Selftest fails (wrong letter) | SNR too low | Increase `n_sequences` or `p300_amplitude_uv` in simulator config |
| `FileNotFoundError: No trained model` | `spell` before `train` | Run `python run.py train` first |
| `serial.SerialException` | Wrong port | Check Device Manager (Win) / `ls /dev/tty*` (Linux/Mac) |
| Low accuracy on real EEG | Bad electrode contact / noise | Apply more gel; move away from power supplies; increase `n_sequences` |

---

## Run the Unit Tests

```bash
cd P300-Analyzer-V1/p300_speller
pytest -v
# Expected: 20 passed
```

In [ ]:
# ── Run the unit tests inside Colab ──────────────────────────────────────
import subprocess, os

_env = {**os.environ, 'SDL_VIDEODRIVER': 'dummy', 'SDL_AUDIODRIVER': 'dummy'}

result = subprocess.run(
    ['python', '-m', 'pytest', 'p300_speller/tests/', '-v'],
    capture_output=True, text=True, env=_env, cwd=REPO_ROOT
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)